# 02 EModel Optimisation

Build an `EModelOptimizationScanConfig`, expand it with `GridScanGenerationTask`,
register the campaign and single-coordinate `TaskConfig` entities in entitycore, and
run the registered single config locally through OBI-One's task runner.

The local path executes `EModelOptimizationTask` directly, without Launch System.
The Launch System submission workflow is retained below as commented reference code.
The local run stages the entity assets, builds the versioned optimisation artifacts,
runs BluePyEModel/NEURON, writes the analysis figures, and exports the SONATA
model package.

**Reads from:** the entitycore staging project — the extraction `TaskResult` entity
from notebook 01, plus `CellMorphology` and `IonChannelModel` entities.

**Writes to:** the local scan output directory under `obi-output/02_emodel_optimization`.

The example uses **`optimiser='SO-CMA'`** with very small `max_ngen=1` and
`offspring_size=2`. Those values only exercise the pipeline; see *Tuning* below for
production settings.

## Task 2 guide

This notebook documents the Task 2 optimisation configuration and handoff for the SSCx C060109A1-SR-C1 dataset via entitycore staging. It assumes that the eFeature extraction stage has already registered the input `TaskResult`; the extraction and export notebooks are not part of this checkout.

### Prerequisites

- `bluepyemodel` and a compatible **NEURON 8.2.x** runtime are required for the local optimisation.
- `nrnivmodl` must be available to compile the selected mechanisms.
- Access to entitycore staging is required; authentication uses `obi_auth.get_token(environment=\"staging\")`.
- Use real staging IDs for the extraction `TaskResult`, `CellMorphology`, `IonChannelModel`, and `ETypeClass` entities.

### Output registration

The optimisation, analysis and export stages run fully locally. Registering the result
entities additionally needs an EntitySDK release that ships the
`entitysdk.registration` helper package (`register_emodel`, `register_memodel`,
`register_emodel_optimization_result`). No published EntitySDK release provides it yet,
so the run cell below reports registration as skipped and leaves every artifact on disk.
Everything upstream of registration still completes, and the same `TaskConfig` can be
re-run or submitted to Launch System once that release is available.

### Workflow and ownership

1. Construct and validate `EModelOptimizationScanConfig`.
2. Expand it with `GridScanGenerationTask`, registering the campaign and single-coordinate `TaskConfig` entities.
3. Run the registered single config locally with `obi.run_task_for_single_config(...)`.
4. The Launch System submission path remains below as commented reference code.

The local runner stages the entity assets, builds the versioned params/recipe artifacts,
and runs BluePyEModel/NEURON. The scientific `TaskConfig` contains optimisation inputs
and settings; the commented Launch System resource profile is not needed for the direct
local run.

### Inputs and outputs

Inputs are entity-based: the extraction `TaskResult`, `CellMorphology`, and selected `IonChannelModel` entities are fetched from entitycore staging. The local run writes the staged morphology, mechanisms, params, recipes, checkpoints, figures, and exports below `obi-output/02_emodel_optimization/grid_scan/0/`.

With `coordinate_directory_option=\"ZERO_INDEX\"`, coordinate 0 is always at `<output_root>/0/`; sweep dimensions add `1/`, `2/`, and so on.

### Tuning

The notebook uses `optimiser=\"SO-CMA\"`, `max_ngen=1`, and `offspring_size=2` for a small, reproducible smoke test; the resulting model is not meant to be a good fit. For production, increase `max_ngen` to approximately 100 and `offspring_size` to approximately 20, and consider sweeping `seed` to fit several models in parallel. `offspring_size` must be at least 2 for the CMA optimisers, because the strategy derives its parent count as `offspring_size // 2`.

Only the mechanisms you assign in `mechanism_regions` become active parameters. The other selected `IonChannelModel` entities are still staged and compiled, so you can add them to a region without changing the mechanism download step.

Setting a block field to a list creates a sweep dimension. Use tuples when a list-valued parameter should remain a fixed value, for example `validation_protocols`.

`OptimizationSettings.optimiser` supports `SO-CMA`, `MO-CMA`, and `IBEA`; the selected value is written to the recipe `pipeline_settings`.

## Imports

In [ ]:
# Optional imports used only by the commented Launch System reference cells below.
# from http import HTTPStatus
# import httpx

import shutil

import obi_one as obi
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token
from obi_one.core.info import Info
from obi_one.scientific.from_id.etype_class_from_id import ETypeClassFromID
from obi_one.scientific.from_id.ion_channel_model_from_id import IonChannelModelFromID
from obi_one.scientific.from_id.task_result_from_id import TaskResultFromID
from obi_one.scientific.tasks.emodel_building.task2_emodel_optimization.blocks import (
    CustomDistanceDependentDistribution,
    GlobalParameterSelection,
    MechanismRegionSelection,
    OptimizationInitialize,
    OptimizationInputs,
    OptimizationParams,
    OptimizationSettings,
    OptimizationValue,
    ParameterSelection,
    ParametersSelection,
)


## Connect to entitycore

The entitycore client is used to register the scan configs and to stage all inputs
for the direct local optimisation run. The Launch System API client is not needed
for the local path; its setup is retained as commented reference code below.

In [ ]:
environment = "staging"
virtual_lab_id = obi.LAB_ID_STAGING_TEST
project_id = obi.PROJECT_ID_STAGING_TEST
token = get_token(environment=environment)

project_context = ProjectContext(
    virtual_lab_id=virtual_lab_id,
    project_id=project_id,
)
db_client = Client(
    api_url="https://staging.cell-a.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)

# Optional Launch System reference setup; uncomment together with the commented
# submission cells below when using the remote/local Launch System workflow.
# OBI_ONE_API_URL = "http://127.0.0.1:8100"
# api_headers = {
#     "Authorization": f"Bearer {token}",
#     "Accept": "application/json",
#     "virtual-lab-id": virtual_lab_id,
#     "project-id": project_id,
# }
# api_client = httpx.Client(base_url=OBI_ONE_API_URL, headers=api_headers)

print("Connected to entitycore staging for local Task 2 execution.")

## Build the scan config

The optimisation stage uses entity-based inputs:
- `target_efeatures`: the extraction `TaskResult` entity from notebook 01
- `morphology`: a `CellMorphology` entity (the SWC/ASC asset is staged by the worker)
- `ion_channel_models`: `IonChannelModel` entities (their `.mod` assets are staged by the worker).

The cell below reproduces the mechanism and parameter layout from the BluePyEModel
L5PC `pyr.json` example: passive parameters under `all`, calcium and potassium
mechanisms across the active regions, sodium and Ih distributions, and the same
regional bounds and fixed values. The compiler writes the legacy grouped params format.


In [ ]:
# Replace with real entity IDs from your staging project.
EXTRACTION_TASK_RESULT_ID = "743c1709-af3e-4433-816c-d38bd994ed70" #"38362e43-04c2-4c1d-8e21-18d7a3750eea"
ETYPE_ID = "75703892-27a4-4588-9f12-145b08051db5"  # Staging: cADpyr 
MORPHOLOGY_ID = "de34e33d-594d-4227-83d1-4e5bb657cf0c"  # C060114A5, species: rat

ION_CHANNEL_MODEL_IDS = {
    "CaDynamics_DC0": "3d371a7e-ff6a-45a0-9f32-34505ac940df",
    "Ca_HVA2": "21ebb7ab-b41b-441d-b494-6665075d26b0",
    "Ca_LVAst": "91cfc99a-9dc7-462c-9b08-330dfb4db871",
    "Ih": "af9eef69-40a3-4e41-9c3f-542a9ff689fc",
    "K_Pst": "bdc5d10e-fd25-403b-83cd-81853676ca35",
    "K_Tst": "8c60dccf-8dfc-48a4-a2db-358b9643e3fb",
    "Nap_Et2": "7decbf75-9414-4754-bf54-5fa787097206",
    "NaTg": "13c947c3-cb76-4a9a-91f4-146e95bd25f3",
    "SK_E2": "e1090418-c607-4901-88ae-a5ceb5bfb75a",
    "SKv3_1": "a014a0f6-217a-4e18-935f-aa298e35c9d1",
}

# Reproduce the mechanism and parameter layout from BluePyEModel's L5PC pyr.json.
# All mechanism assets are still resolved by their EntityCore IDs.
ION_CHANNEL_MODELS = {
    name: IonChannelModelFromID(id_str=entity_id)
    for name, entity_id in ION_CHANNEL_MODEL_IDS.items()
}

def fixed(value):
    return ParameterSelection(value=OptimizationValue(value=value))


def bounds(lower, upper, distribution="uniform"):
    return ParameterSelection(
        value=OptimizationValue(mode="bounds", bounds=(lower, upper)),
        distribution=distribution,
    )


def channel(name, parameters=None):
    return MechanismRegionSelection(
        ion_channel_model=ION_CHANNEL_MODELS[name],
        parameters=parameters or {},
    )

scan_config = obi.EModelOptimizationScanConfig(
    info=Info(
        campaign_name="L5PC Optimisation",
        campaign_description="Optimise L5PC model against extracted e-features.",
    ),
    initialize=OptimizationInitialize(
        emodel="L5PC",
        etype=ETypeClassFromID(id_str=ETYPE_ID),
    ),
    inputs=OptimizationInputs(
        target_efeatures=TaskResultFromID(id_str=EXTRACTION_TASK_RESULT_ID),
        morphology=obi.CellMorphologyFromID(id_str=MORPHOLOGY_ID),
    ),
    parameters_selection=ParametersSelection(
        ion_channel_models=[
            IonChannelModelFromID(id_str=icm_id)
            for icm_id in ION_CHANNEL_MODEL_IDS.values()
        ],
        global_parameters={
            "v_init": GlobalParameterSelection(value=OptimizationValue(value=-80.0)),
            "celsius": GlobalParameterSelection(value=OptimizationValue(value=34.0)),
        },
        base_parameters={
            "myelinated": {"cm": fixed(0.02)},
            "all": {
                "Ra": fixed(100.0),
                "g_pas": bounds(1e-5, 6e-5),
                "e_pas": bounds(-95.0, -60.0),
            },
            "axonal": {"cm": fixed(1.0), "ena": fixed(50.0), "ek": fixed(-90.0)},
            "somatic": {"cm": fixed(1.0), "ena": fixed(50.0), "ek": fixed(-90.0)},
            "apical": {"cm": fixed(2.0), "ena": fixed(50.0), "ek": fixed(-90.0)},
            "basal": {"cm": fixed(2.0)},
        },
        mechanism_regions={
            "allact": (
                channel("CaDynamics_DC0"),
                channel("Ca_HVA2"),
                channel("Ca_LVAst"),
            ),
            "somaxon": (
                channel("SKv3_1"),
                channel("SK_E2"),
                channel("K_Pst"),
                channel("K_Tst"),
            ),
            "axonal": (
                channel("CaDynamics_DC0", {
                    "decay": bounds(20.0, 300.0),
                    "gamma": bounds(0.005, 0.05),
                }),
                channel("NaTg", {
                    "vshifth": fixed(15.0),
                    "slopem": fixed(7.0),
                    "gNaTgbar": bounds(0.0, 1.5),
                }),
                channel("Nap_Et2", {"gNap_Et2bar": bounds(0.0, 0.02)}),
                channel("K_Pst", {"gK_Pstbar": bounds(0.0, 1.0)}),
                channel("K_Tst", {"gK_Tstbar": bounds(0.0, 0.2)}),
                channel("SKv3_1", {"gSKv3_1bar": bounds(0.0, 1.0)}),
                channel("Ca_HVA2", {"gCa_HVAbar": bounds(0.0, 0.001)}),
                channel("Ca_LVAst", {"gCa_LVAstbar": bounds(0.0, 0.01)}),
                channel("SK_E2", {"gSK_E2bar": bounds(0.0, 0.1)}),
            ),
            "somatic": (
                channel("CaDynamics_DC0", {
                    "decay": bounds(20.0, 300.0),
                    "gamma": bounds(0.005, 0.05),
                }),
                channel("NaTg", {
                    "vshiftm": fixed(13.0),
                    "vshifth": fixed(15.0),
                    "slopem": fixed(7.0),
                    "gNaTgbar": bounds(0.0, 0.3),
                }),
                channel("K_Pst", {"gK_Pstbar": bounds(0.0, 0.2)}),
                channel("K_Tst", {"gK_Tstbar": bounds(0.0, 0.1)}),
                channel("SKv3_1", {"gSKv3_1bar": bounds(0.0, 1.0)}),
                channel("Ca_HVA2", {"gCa_HVAbar": bounds(0.0, 0.001)}),
                channel("Ca_LVAst", {"gCa_LVAstbar": bounds(0.0, 0.01)}),
                channel("SK_E2", {"gSK_E2bar": bounds(0.0, 0.1)}),
            ),
            "apical": (
                channel("NaTg", {
                    "vshiftm": fixed(6.0),
                    "vshifth": fixed(6.0),
                    "gNaTgbar": bounds(0.0, 0.1, "decay"),
                }),
                channel("SKv3_1", {"gSKv3_1bar": bounds(0.0, 0.003)}),
                channel("Ca_HVA2", {"gCa_HVAbar": bounds(0.0, 0.0001)}),
                channel("Ca_LVAst", {"gCa_LVAstbar": bounds(0.0, 0.001)}),
            ),
            "somadend": (
                channel("Ih", {"gIhbar": bounds(0.0, 0.0002, "exp")}),
            ),
            "basal": (
                channel("CaDynamics_DC0", {"gamma": bounds(0.005, 0.05)}),
                channel("Ca_HVA2", {"gCa_HVAbar": bounds(0.0, 0.0001)}),
                channel("Ca_LVAst", {"gCa_LVAstbar": bounds(0.0, 0.001)}),
            ),
        },
        distribution_parameters={
            "decay": {
                "constant": OptimizationValue(mode="bounds", bounds=(-0.1, 0.0)),
            },
        },
    ),
    # Custom distance-dependent distribution declarations.
    # Standard distributions such as "uniform" are available globally and are not
    # declared in this custom-only field.
    distance_dependent_distributions={
        "decay": CustomDistanceDependentDistribution(
            name="decay",
            function="math.exp({distance}*{constant})*{value}",
            parameters=["constant"],
        ),
    },
    optimization_settings=OptimizationSettings(
        optimiser="SO-CMA",
        max_ngen=1,
        optimisation_timeout=300.0,
        validation_threshold=5.0,
        seed=1,
    ),
    optimization_params=OptimizationParams(
        offspring_size=2,
    ),
)
print(scan_config.optimization_settings.to_dict(scan_config.optimization_params))
print(f"Configured mechanism regions: {list(scan_config.parameters_selection.mechanism_regions)}")
print(f"Configured distributions: {list(scan_config.distance_dependent_distributions)}")

## Optional Launch System resource profile

The following metadata is only needed when using the commented Launch System path.
It is not used by the direct local execution below.

In [ ]:
# Optional Launch System metadata.
# LAUNCH_RESOURCE_PROFILE = {
#     "builtin_task": "emodel_optimisation",
#     "compute_cell": "cell_a",
#     "instances": 1,
#     "instance_type": "small",
#     "timelimit": "02:00",
#     "workers": 1,
# }
# print(LAUNCH_RESOURCE_PROFILE)

## Register the campaign and single TaskConfigs

`GridScanGenerationTask.execute()` expands the scan and registers the campaign and
coordinate configs in entitycore. Before registration, the next cell removes only
the previous local `obi-output/02_emodel_optimization/grid_scan` directory; it does
not delete EntityCore entities or Launch System execution records.
It does not run the optimisation.

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../../../../obi-output/02_emodel_optimization/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
output_root = grid_scan.output_root_absolute
if output_root.parts[-3:] != ("obi-output", "02_emodel_optimization", "grid_scan"):
    raise RuntimeError(f"Refusing to clean unexpected Task 2 output root: {output_root}")
if output_root.exists():
    shutil.rmtree(output_root)
    print(f"Deleted previous local Task 2 execution: {output_root}")
else:
    print(f"No previous local Task 2 execution found: {output_root}")
grid_scan.execute(db_client=db_client)

campaign_entity = grid_scan.form.campaign
single_config_entity = grid_scan.single_configs[0].single_entity
campaign_id = campaign_entity.id
single_config_id = single_config_entity.id

print(f"Registered campaign TaskConfig: {campaign_id}")
print(f"Registered {len(grid_scan.single_configs)} single TaskConfig(s).")
for config in grid_scan.single_configs:
    print(f"  Coordinate {config.idx}: {config.single_entity.id}")

In [ ]:
registered_config = db_client.get_entity(
    entity_type=models.TaskConfig,
    entity_id=single_config_id,
)
print(
    f"Selected '{registered_config.__class__.__name__}({registered_config.task_config_type})'"
    f" with ID {registered_config.id}"
)

## Run the optimisation locally

The direct local runner bypasses Launch System and executes the registered single
configuration in this notebook process: it stages the entity assets, compiles the
mechanisms, runs BluePyEModel/NEURON, writes the analysis figures and exports the
SONATA model package.

For a sweep, this runs every coordinate sequentially. Keep `max_ngen` and
`offspring_size` small for a smoke test: with `max_ngen=1` this still evaluates every
protocol in the extracted feature set and takes several minutes.

Output-entity registration needs the `entitysdk.registration` helpers described above.
Until an EntitySDK release provides them, the cell reports registration as skipped and
prints the artifacts written for each coordinate.

In [ ]:
from pathlib import Path

# Raised by EModelOptimizationTask.register_output_entities when the installed
# EntitySDK has no entitysdk.registration helper package.
REGISTRATION_MARKER = "entitysdk.registration"


def summarise_coordinate(coordinate_root: Path) -> None:
    """Print the artifacts a finished coordinate produced."""
    for label, pattern in (
        ("params", "config/params/params.json"),
        ("recipe", "config/recipes.json"),
        ("checkpoint", "checkpoints/**/*.pkl"),
        ("final model", "final.json"),
        ("figure", "figures/**/*.pdf"),
        ("sonata export", "export_emodels_sonata/**/nodes.h5"),
    ):
        found = sorted(coordinate_root.glob(pattern))
        status = f"{len(found)} file(s)" if found else "missing"
        print(f"    {label:<14} {status}")


local_results = []
registration_skipped = False

for single_config in grid_scan.single_configs:
    coordinate_root = Path(single_config.coordinate_output_root).resolve()
    try:
        result = obi.run_task_for_single_config(
            single_config,
            db_client=db_client,
            entity_cache=False,
        )
        local_results.append(result)
        print(f"Coordinate {single_config.idx}: completed and registered -> {result}")
    except RuntimeError as exc:
        if REGISTRATION_MARKER not in str(exc):
            raise
        registration_skipped = True
        local_results.append(coordinate_root)
        print(f"Coordinate {single_config.idx}: optimisation, analysis and export completed.")
        print(f"  Output root: {coordinate_root}")
    summarise_coordinate(coordinate_root)

if registration_skipped:
    print(
        "\nResult entities were not registered: the installed EntitySDK does not "
        "provide the entitysdk.registration helpers. All artifacts above are on disk."
    )


## Optional Launch System execution (commented)

The original Launch System submission path is preserved below. Uncomment this
cell and the Launch System imports/setup above when you want to submit the
registered config instead of running it locally.

In [ ]:
# payload = {
#     "task_type": "emodel_optimization",
#     "config_id": str(single_config_id),
# }
# response = api_client.post(
#     url="/declared/task/launch",
#     json=payload,
#     timeout=300.0,
# )
#
# if response.status_code != HTTPStatus.OK:
#     print(f"Launch failed ({response.status_code}): {response.text}")
#     response.raise_for_status()
#
# launch_info = response.json()
# activity_id = launch_info["activity_id"]
# job_id = launch_info["job_id"]
# print("Launch submitted successfully.")
# print(f"  activity_id: {activity_id}")
# print(f"  job_id:      {job_id}")

In [ ]:
# Optional Launch System activity polling reference.
# execution_activity = db_client.get_entity(
#     entity_type=models.TaskActivity,
#     entity_id=activity_id,
# )
# print(f"Activity status: {execution_activity.status}")
# print(f"Activity execution ID: {execution_activity.execution_id}")
# print("The optimisation result will be registered after the remote cluster job completes.")